# 6) Unit commitment: on/off decisions, MILP and the pricing problem

## Objectives

At the end of this notebook you should be able to:

- explain why minimum stable output and start-up costs cannot be modelled with continuous variables alone
- formulate **unit commitment** as a **mixed-integer linear program (MILP)** in Linopy, with binary on/off variables, start-up variables and the constraints that link them
- measure the **integrality gap** between the MILP and its LP relaxation, and explain what the relaxation gets wrong
- run the same model with PyPSA's committable generators — and avoid the trap that it reports **zero prices** after a MILP
- compute prices from the **fixed-commitment LP**, and show why they can leave a plant unable to recover its start-up cost (**uplift**)

So far every decision could take any value between bounds. Real thermal plants cannot: a coal unit is either off, or running between its **minimum stable output** and its maximum, and starting it costs fuel, wear and time. That makes the decisions discrete — and changes what a price means. Same sunny day as lessons 3 and 4.

In [1]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### The day, with technical minimums and start-up costs

In [2]:
T = 24
hours = np.arange(T)
load = 1900 + 400 * np.sin(np.pi * (hours - 6) / 12).clip(0) + 250 * np.exp(-((hours - 19) / 2.0) ** 2)
free = 2200 * 0.8 * np.clip(np.sin(np.pi * (hours - 6) / 13), 0, None) + 400.0
plants = pd.DataFrame({"p_nom": [900, 900, 500], "mc": [60, 95, 150],
                       "p_min_pu": [0.4, 0.3, 0.1],                 # minimum stable output, share of p_nom
                       "start_up_cost": [20_000, 8_000, 1_000],     # EUR per start
                       "on_before": [1, 0, 0]},                     # coal was running before the day starts
                      index=pd.Index(["coal", "ccgt", "ocgt"], name="gen"))
plants.assign(p_min_MW=plants.p_nom * plants.p_min_pu)

      p_nom   mc  p_min_pu  start_up_cost  on_before  p_min_MW
gen                                                           
coal    900   60       0.4          20000          1     360.0
ccgt    900   95       0.3           8000          0     270.0
ocgt    500  150       0.1           1000          0      50.0

Lesson 3's unconstrained LP ran coal at 139 MW at midday. With a minimum stable output of 360 MW, that operating point does not exist: coal is either off (0 MW) or between 360 and 900 MW. The set of allowed outputs has a **gap**, and no linear constraint on a continuous variable can describe a gap.

### The MILP formulation

Add a **binary** variable u[g, t] ∈ {0, 1} (on or off) and a start-up variable s[g, t] ≥ 0 per plant and hour:

- **capacity linked to status**: p_min · u[g, t] ≤ g[g, t] ≤ p_max · u[g, t] — off means exactly 0, on means between the limits
- **start-ups**: s[g, t] ≥ u[g, t] − u[g, t−1], with the status before the day given (`on_before`)
- **objective**: Σ marginal cost · g + Σ start-up cost · s

Because s is minimised and costs money, it takes the value 1 exactly when a unit switches on.

In [3]:
def unit_commitment(relax=False):
    """Own Linopy model of the day. relax=True replaces the binaries by continuous 0..1 variables."""
    m = linopy.Model()
    t, G = pd.RangeIndex(T, name="t"), plants.index
    g = m.add_variables(lower=0, coords=[t, G], name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(free, coords=[t]), name="free")
    u = (m.add_variables(lower=0, upper=1, coords=[t, G], name="u") if relax
         else m.add_variables(coords=[t, G], name="u", binary=True))
    s = m.add_variables(lower=0, coords=[t, G], name="start")
    p_max = xr.DataArray(plants.p_nom.to_numpy(), coords=[G])
    p_min = xr.DataArray((plants.p_nom * plants.p_min_pu).to_numpy(), coords=[G])
    m.add_constraints(g - p_max * u <= 0, name="max_when_on")
    m.add_constraints(g - p_min * u >= 0, name="min_when_on")
    m.add_constraints(s - u + u.shift(t=1) >= 0, name="start_up", mask=xr.DataArray(hours > 0, coords=[t]))
    m.add_constraints(s.sel(t=0) - u.sel(t=0) >= -xr.DataArray(plants.on_before.to_numpy(), coords=[G]), name="start_up_0")
    m.add_constraints(g.sum("gen") + f == xr.DataArray(load, coords=[t]), name="balance")
    m.add_objective((g * xr.DataArray(plants.mc.to_numpy(), coords=[G])).sum()
                    + (s * xr.DataArray(plants.start_up_cost.to_numpy(), coords=[G])).sum())
    m.solve(solver_name="highs", output_flag=False)
    return m


milp = unit_commitment()
status = milp.variables["u"].solution.to_pandas().round().astype(int)
dispatch = milp.variables["g"].solution.to_pandas()
curtailed = float((free - milp.variables["free"].solution).sum())
print(f"MILP cost {milp.objective.value:,.2f} EUR, curtailed solar {curtailed:,.0f} MWh\n")
pd.concat({"on": status.T, "MW": dispatch.T.round(0).astype(int)})

MILP cost 1,880,184.24 EUR, curtailed solar 727 MWh



t         0    1    2    3    4    5    6    7    8    9   ...   14   15   16  \
   gen                                                     ...                  
on coal    1    1    1    1    1    1    1    1    1    1  ...    1    1    1   
   ccgt    1    1    1    1    1    1    1    1    0    0  ...    0    0    0   
   ocgt    0    0    0    0    0    0    0    0    0    0  ...    0    0    0   
MW coal  900  900  900  900  900  900  900  900  882  616  ...  360  360  559   
   ccgt  600  600  600  600  600  600  600  282    0    0  ...    0    0    0   
   ocgt    0    0    0    0    0    0    0    0    0    0  ...    0    0    0   

t         17   18   19   20   21   22   23  
   gen                                      
on coal    1    1    1    1    1    1    1  
   ccgt    0    1    1    1    1    1    1  
   ocgt    0    0    0    0    0    0    0  
MW coal  878  900  900  900  900  900  900  
   ccgt    0  374  850  795  692  626  605  
   ocgt    0    0    0    0    0    

Read the schedule row by row. **Coal stays on all day**: at midday it drops to exactly its 360 MW minimum for five hours (11:00–15:59) rather than switching off, because a restart would cost 20,000 € and the evening needs it at full output again. Running at its minimum while the sun shines means there is more free energy than demand — **727 MWh of solar is curtailed**. The minimum stable output does what the ramp limit did in lesson 3: an inflexible plant pushes clean energy out. The **CCGT** runs through the night, shuts down at 08:00 and starts again at 18:00 for the evening peak. It was off before the day began, so the night run also counts as a start: two starts, 16,000 €. The OCGT is never needed.

### What the LP relaxation gets wrong

Replace u ∈ {0, 1} by 0 ≤ u ≤ 1 and you get the **LP relaxation** — the model a solver solves first inside its branch-and-bound search:

In [4]:
relaxed = unit_commitment(relax=True)
gap = milp.objective.value - relaxed.objective.value
print(f"MILP {milp.objective.value:,.0f} EUR | LP relaxation {relaxed.objective.value:,.0f} EUR | "
      f"integrality gap {gap:,.0f} EUR ({gap / milp.objective.value:.2%})")
relaxed.variables["u"].solution.to_pandas().T.round(2) + 0.0

MILP 1,880,184 EUR | LP relaxation 1,845,721 EUR | integrality gap 34,464 EUR (1.83%)


t       0     1     2     3     4     5     6     7     8     9   ...    14  \
gen                                                               ...         
coal  1.00  1.00  1.00  1.00  1.00  1.00  1.00  1.00  0.98  0.68  ...  0.39   
ccgt  0.67  0.67  0.67  0.67  0.67  0.67  0.67  0.31  0.00  0.00  ...  0.00   
ocgt  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.00  ...  0.00   

t       15    16    17    18    19    20    21   22    23  
gen                                                        
coal  0.39  0.98  0.98  1.00  1.00  1.00  1.00  1.0  1.00  
ccgt  0.00  0.00  0.00  0.94  0.94  0.88  0.77  0.7  0.67  
ocgt  0.00  0.00  0.00  0.00  0.00  0.00  0.00  0.0  0.00  

[3 rows x 24 columns]

The relaxation is 34,464 € (1.83 %) cheaper — because it is allowed to do things no plant can do. At midday it runs coal at **u = 0.39**: "39 % switched on", which lowers the minimum output to 0.4 × 900 × 0.39 = 140 MW and lets coal produce what lesson 3's unconstrained LP wanted. At night it keeps the CCGT 67 % on and pays only 67 % of its start-up cost. These fractional states are meaningless physically, which is why the binaries matter. A MILP solver starts from exactly this relaxation and then **branches** on fractional variables (u = 0 or u = 1) until it proves an integer solution optimal; the gap measures how much work that is, and how wrong an LP-only model of these plants would be.

### The same model in PyPSA — and a silent trap

PyPSA writes this formulation for you when a generator is `committable=True`, with `p_min_pu` and `start_up_cost`. The status before the day is given by `up_time_before` / `down_time_before` (hours the unit has been on or off); a long history means "no constraint from the past".

In [5]:
def pypsa_day(committable=True, status_fixed=None):
    n = pypsa.Network()
    n.set_snapshots(hours)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=load)
    n.add("Generator", "free", bus="HR", p_nom=free.max(), p_max_pu=free / free.max(), marginal_cost=0)
    for gen, r in plants.iterrows():
        if status_fixed is None:
            n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc, committable=committable,
                  p_min_pu=r.p_min_pu, start_up_cost=r.start_up_cost,
                  up_time_before=24 if r.on_before else 0, down_time_before=0 if r.on_before else 24)
        else:                                   # fixed commitment: the bounds follow the given status
            on = status_fixed[gen].to_numpy()
            n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc,
                  p_max_pu=on, p_min_pu=on * r.p_min_pu)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n


n_uc = pypsa_day()
assert np.isclose(n_uc.objective, milp.objective.value, rtol=1e-9)
assert (n_uc.generators_t.status[plants.index].astype(int).to_numpy() == status.to_numpy()).all()
print(f"PyPSA unit commitment: {n_uc.objective:,.2f} EUR, same on/off schedule as the own MILP")
print("PyPSA's marginal prices after the MILP:", n_uc.buses_t.marginal_price["HR"].round(2).unique())

PyPSA unit commitment: 1,880,184.24 EUR, same on/off schedule as the own MILP
PyPSA's marginal prices after the MILP: [0.]


Same cost, same schedule — and **every price is 0.0**. A MILP has no dual values: once a variable must be 0 or 1, there is no smooth trade-off at the margin for a dual to measure. The solver returns none, and PyPSA fills the price table with zeros **without any warning**. Any analysis that reads `n.buses_t.marginal_price` after a unit-commitment run is silently wrong.

### Prices from the fixed-commitment LP

The standard remedy, used in many market-clearing and research models: take the optimal on/off schedule, **fix** it, and solve the remaining LP — the outputs are still continuous, so the balance constraint has a dual again. In PyPSA, fixing means turning each unit's status into hourly bounds (`p_max_pu = status`, `p_min_pu = status × minimum`):

In [6]:
n_fixed = pypsa_day(status_fixed=status)
before = plants.on_before.to_numpy()
starts = np.maximum(np.diff(np.vstack([before, status.to_numpy()]), axis=0), 0)
start_up_costs = float((starts * plants.start_up_cost.to_numpy()).sum())
assert np.isclose(n_fixed.objective + start_up_costs, milp.objective.value, rtol=1e-9), "same total cost"
price = n_fixed.buses_t.marginal_price["HR"].to_numpy() + 0.0
print(f"fixed LP {n_fixed.objective:,.2f} + start-ups {start_up_costs:,.0f} = MILP {milp.objective.value:,.2f} EUR")
pd.Series(price, index=hours, name="price_EUR_MWh").to_frame().T

fixed LP 1,864,184.24 + start-ups 16,000 = MILP 1,880,184.24 EUR


                 0     1     2     3     4     5     6     7     8     9   \
price_EUR_MWh  95.0  95.0  95.0  95.0  95.0  95.0  95.0  95.0  60.0  60.0   

               ...   14   15    16    17    18    19    20    21    22    23  
price_EUR_MWh  ...  0.0  0.0  60.0  60.0  95.0  95.0  95.0  95.0  95.0  95.0  

[1 rows x 24 columns]

The fixed-commitment LP gives the same total cost once the 16,000 € of start-ups are added, and now every hour has a price: 95 €/MWh at night (the CCGT is marginal), 60 €/MWh in the morning and afternoon (coal), and **0 €/MWh from 11:00 to 15:59**. At midday coal is fixed on and cannot go below its minimum, so the marginal MWh comes from curtailed solar — worth nothing. The same price pattern as lesson 3's ramp-limited day, from a different cause.

### Who recovers their costs? Uplift

In [7]:
profit = {}
for i, gen in enumerate(plants.index):
    output = n_fixed.generators_t.p[gen].to_numpy()
    margin = float(((price - plants.loc[gen, "mc"]) * output).sum())
    profit[gen] = {"market_margin": margin, "start_up_costs": float(starts[:, i].sum() * plants.loc[gen, "start_up_cost"]),
                   "starts": int(starts[:, i].sum())}
profit = pd.DataFrame(profit).T
profit["profit"] = profit.market_margin - profit.start_up_costs
profit["uplift_needed"] = (-profit.profit).clip(lower=0) + 0.0
profit.round(0)

      market_margin  start_up_costs  starts    profit  uplift_needed
coal       333000.0             0.0     0.0  333000.0            0.0
ccgt            0.0         16000.0     2.0  -16000.0        16000.0
ocgt            0.0             0.0     0.0       0.0            0.0

Coal earns a 333,000 € margin over the day — the night and evening hours at 95 €/MWh pay for its midday hours at 0. The **CCGT earns nothing above its fuel cost**: whenever it runs, it is the marginal plant and the price equals its own marginal cost of 95 €/MWh. Its two starts cost 16,000 €, so at these prices it **loses 16,000 €** — even though running it is part of the cheapest schedule for the system. Paid only the market price, no owner would start it.

Markets deal with this in two ways. Many US markets pay the shortfall as an **uplift** (make-whole) payment on top of the market price, funded by consumers. The European day-ahead market (M2.3) instead lets sellers submit **block orders** — all-or-nothing offers across several hours — so the start-up cost is part of the offer itself and an accepted block covers it. Either way, the lesson is the same: with non-convex costs such as start-ups, **marginal prices alone do not cover every plant's costs**, and some payment beyond the energy price is needed.

## Check your understanding

**Formulation**

1. Why does coal stay on all day, even though it produces at its minimum during five hours of curtailed solar? Estimate what switching it off and on again would cost and save.
2. What would happen to the schedule if coal's minimum stable output were 0 %? Predict, then compare with lesson 3.

**Prices**

1. Why can a MILP not provide dual values? What exactly is "fixed" in the fixed-commitment LP, and why does that make the dual meaningful again?
2. The CCGT needs an uplift. In a market without uplift payments, how would its owner change its bids — and what would that do to the evening price?